# Computing Word-Level Reading Measures from Eye-Tracking Data

This tutorial demonstrates how to compute AOI-based word-level reading measures from eye-tracking data using predefined Areas of Interest (AOIs).

## What you will learn

In this tutorial, you will learn how to:

- load a DataFrame containing fixation events,
- load a DataFrame defining word-level AOIs with their bounding boxes,
- map fixations to the corresponding AOIs,
- compute word-level reading measures using `compute_reading_measures`, and
- inspect the resulting DataFrame of computed reading measures.

In [1]:
import polars as pl

from pymovements import Dataset, Events
from pymovements.measure.reading.processing import compute_reading_measures
from pymovements.stimulus.text import TextStimulus

/home/antonia/Dokumente/Arbeit/pymovements/.venv/lib64/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


We begin by loading a dataset containing fixation events together with the corresponding AOI definitions. In this tutorial, we use the `GGTG` dataset, which can be loaded as follows:

In [2]:
dataset = Dataset('GGTG', path='data/GGTG')

# Download the dataset and extract all archives.
dataset.download()

# Load the dataset into memory for processing
dataset.load(subset={'subject_id': 'P01'})

INFO:pymovements.dataset.dataset:
        You are downloading the Gaze-Guided Text Generation. Please be aware that pymovements does not
        host or distribute any dataset resources and only provides a convenient interface to
        download the public dataset resources that were published by their respective authors.

        Please cite the referenced publication if you intend to use the dataset in your research.
        


Verifying existing file: data/GGTG/downloads/samples.zip
Using existing verified file: data/GGTG/downloads/samples.zip
Verifying existing file: data/GGTG/downloads/fixations.zip
Using existing verified file: data/GGTG/downloads/fixations.zip
Verifying existing file: data/GGTG/downloads/measures.zip
Using existing verified file: data/GGTG/downloads/measures.zip
Verifying existing file: data/GGTG/downloads/aoi-csvs.zip
Using existing verified file: data/GGTG/downloads/aoi-csvs.zip
Extracting samples.zip to data/GGTG/raw


Extracting archive: 100%|██████████| 24/24 [00:02<00:00, 10.20file/s]


Extracting fixations.zip to data/GGTG/precomputed_events


Extracting archive: 100%|██████████| 24/24 [00:00<00:00, 470.00file/s]


Extracting measures.zip to data/GGTG/precomputed_reading_measures


Extracting archive: 100%|██████████| 24/24 [00:00<00:00, 741.02file/s]


Extracting aoi-csvs.zip to data/GGTG/stimuli


Loading gaze files: 100%|██████████| 1/1 [00:00<00:00,  4.00file/s]
/home/antonia/Dokumente/Arbeit/pymovements/.venv/lib64/python3.13/site-packages/pymovements/dataset/dataset.py:580: ExperimentalWarning: Stimulus support is experimental. Names and behavior may change without being considered a breaking change. Please set the used pymovements version explicitly to prevent unexptected changes. The used pymovements version is v0.27.1+post31.66cda7a1.
  warn(


For simplicity, we restrict the analysis to a single subject and a single stimulus. Specifically, we use the first subject and the stimulus `goldfish-pos.text.0`. We then select only the required columns and add a column indicating the event type:

In [3]:
stimulus = "goldfish-pos.text.0"
sample_fixation_path = dataset.paths.precomputed_events / \
    dataset.fileinfo['precomputed_events']['filepath'][0]

fixations = pl.read_csv(sample_fixation_path)
fixations = fixations.filter(pl.col('stimulus') == stimulus)
fixations = fixations[['onset',
                       'offset',
                       'duration',
                       'location_x',
                       'location_y',
                       ]]
# add name column for the type of event
fixations = fixations.with_columns(name=pl.lit('fixation'))
fixations.head()

onset,offset,duration,location_x,location_y,name
i64,i64,i64,f64,f64,str
2477683,2478319,636,37.023234,72.70438,"""fixation"""
2478323,2478518,195,105.209184,75.790867,"""fixation"""
2478520,2478685,165,142.643976,74.930482,"""fixation"""
2478688,2478797,109,193.465455,71.871182,"""fixation"""
2478821,2478955,134,96.474074,206.021111,"""fixation"""


In [4]:
sample_fixation_path

PosixPath('data/GGTG/precomputed_events/cleaned/P01.csv')

Next, we load the CSV file containing the AOI definitions for the selected stimulus:

In [5]:
stimulus_rel_path = dataset.fileinfo['textstimulus'].filter(
    pl.col('stimulus') == stimulus).filter(
        pl.col('unit') == 'word')['filepath'][0]
aoi_path = dataset.paths.stimuli / stimulus_rel_path

aoi_df = pl.read_csv(aoi_path, separator=',')
aoi_df = aoi_df.with_columns(aoi_index=pl.col('index'))
aoi_df.head()

index,left,top,right,bottom,section,line,content,aoi_index
i64,f64,f64,f64,f64,str,i64,str,i64
0,43.03125,50.0,102.375,120.0,null,0,"""One""",0
1,102.375,50.0,157.078125,120.0,null,0,"""Last""",1
2,157.078125,50.0,223.828125,120.0,null,0,"""Wish""",2
3,43.40625,190.0,127.921875,260.0,null,2,"""Emma""",3
4,127.921875,190.0,169.65625,260.0,null,2,"""sat""",4


To use the AOI definitions, we convert the DataFrame into a `TextStimulus`:

In [6]:
aoi_text_stimulus = TextStimulus(
    aoi_df,
    aoi_column='content',
    start_x_column='left',
    start_y_column='top',
    end_x_column='right',
    end_y_column='bottom',)
aoi_text_stimulus

index,left,top,right,bottom,section,line,content,aoi_index
i64,f64,f64,f64,f64,str,i64,str,i64
0,43.03125,50.0,102.375,120.0,null,0,"""One""",0
1,102.375,50.0,157.078125,120.0,null,0,"""Last""",1
2,157.078125,50.0,223.828125,120.0,null,0,"""Wish""",2
3,43.40625,190.0,127.921875,260.0,null,2,"""Emma""",3
4,127.921875,190.0,169.65625,260.0,null,2,"""sat""",4
…,…,…,…,…,…,…,…,…
60,112.28125,750.0,192.453125,820.0,null,10,"""taking""",60
61,192.453125,750.0,248.96875,820.0,null,10,"""care""",61
62,248.96875,750.0,279.3125,820.0,null,10,"""of""",62


Next, we map the fixation events to their corresponding AOIs. To do so, we create an `Events` DataFrame and use the `map_to_aois` function:

In [7]:
events = Events(data=fixations)
events.map_to_aois(aoi_text_stimulus)
events.frame

Mapping events to AOIs: 100%|███████████████| 94/94 [00:00<00:00, 707.20event/s]


onset,offset,duration,location_x,location_y,name,index,left,top,right,bottom,section,line,content,aoi_index
i64,i64,i64,f64,f64,str,i64,f64,f64,f64,f64,str,i64,str,i64
2477683,2478319,636,37.023234,72.70438,"""fixation""",null,null,null,null,null,null,null,null,null
2478323,2478518,195,105.209184,75.790867,"""fixation""",1,102.375,50.0,157.078125,120.0,null,0,"""Last""",1
2478520,2478685,165,142.643976,74.930482,"""fixation""",1,102.375,50.0,157.078125,120.0,null,0,"""Last""",1
2478688,2478797,109,193.465455,71.871182,"""fixation""",2,157.078125,50.0,223.828125,120.0,null,0,"""Wish""",2
2478821,2478955,134,96.474074,206.021111,"""fixation""",3,43.40625,190.0,127.921875,260.0,null,2,"""Emma""",3
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2497064,2497292,228,142.362445,365.492969,"""fixation""",25,43.875,330.0,191.953125,400.0,null,4,"""movements""",25
2497308,2497430,122,263.068293,362.665366,"""fixation""",26,191.953125,330.0,303.625,400.0,null,4,"""sluggish.""",26
2497495,2497646,151,845.317105,867.044868,"""fixation""",null,null,null,null,null,null,null,null,null


With all required data prepared, we can now compute the reading measures using `compute_reading_measures`:

In [9]:
def compute_reading_measures(
        fixations: pl.DataFrame,
        aois: pl.DataFrame,
        *,
        word_index_column: str = 'word_idx',
        word_column: str = 'word',
) -> pl.DataFrame:
    """Compute reading measures from fixation sequences.

    This function expects fixations annotated with AOI data. See
    :py:meth:`~pymovements.Events.map_to_aois` for further details.

    Parameters
    ----------
    fixations : pl.DataFrame
        DataFrame with fixation data, containing the column specified by ``word_index_column``.
    aois : pl.DataFrame
        DataFrame with AOI data, containing the columns specified by ``word_index_column`` and
        ``word_column``.
    word_index_column : str
        Shared column name in ``fixations`` and ``aois`` that corresponds to the word index of the
        text.
        (default: ``'aoi'``)
    word_column : str
        Column in ``aois`` with the content within each AOI.
        (default: ``'word'``)

    Returns
    -------
    pl.DataFrame
        DataFrame with computed reading measures.
    """
    # Append an extra dummy fixation to have the next fixation for the actual last fixation.
    dummy_fixation_dict: dict[str, list[int] | list[str]] = {}
    for col, dtype in fixations.schema.items():
        if dtype == pl.String:
            dummy_fixation_dict[col] = ['']
        else:
            dummy_fixation_dict[col] = [0]
    dummy_fixation = pl.DataFrame(
        dummy_fixation_dict,
        schema=fixations.schema,
    )
    fixations = pl.concat([fixations, dummy_fixation])

    # Adjust AOI indices (fix off by one error).
    aois = aois.with_columns(
        (pl.col(word_index_column) - 1).alias(word_index_column),
    )

    # Get original words of the text and their indices.
    word_indices = aois[word_index_column].to_list()
    words = aois[word_column].to_list()

    # Initialize dictionary for reading measures per word.
    rm_dict = {
        word_index: {
            'word': word,
            'word_index': word_index,
            'FFD': 0, 'SFD': 0, 'FD': 0, 'FPRT': 0, 'FPFC': 0, 'FRT': 0, 'TFT': 0, 'RRT': 0,
            'RPD_inc': 0, 'RPD_exc': 0, 'RBRT': 0, 'Fix': 0, 'FPF': 0, 'RR': 0,
            'FPReg': 0, 'TRC_out': 0, 'TRC_in': 0, 'SL_in': 0, 'SL_out': 0, 'TFC': 0,
        } for word_index, word in zip(word_indices, words)
    }

    # Add a catch-all entry for the dummy fixation and invalid AOIs
    rm_dict[-1] = {
        'word': None, 'word_index': -1,
        'FFD': 0, 'SFD': 0, 'FD': 0, 'FPRT': 0, 'FPFC': 0, 'FRT': 0, 'TFT': 0, 'RRT': 0,
        'RPD_inc': 0, 'RPD_exc': 0, 'RBRT': 0, 'Fix': 0, 'FPF': 0, 'RR': 0,
        'FPReg': 0, 'TRC_out': 0, 'TRC_in': 0, 'SL_in': 0, 'SL_out': 0, 'TFC': 0,
    }

    # Variables to track fixation progress.
    right_most_word, cur_fix_word_idx, next_fix_word_idx, next_fix_dur = -1, -1, -1, -1

    # Iterate over fixation data.
    for fixation in fixations.to_dicts():
        try:
            aoi = int(fixation[word_index_column]) - 1
            if aoi not in rm_dict:
                continue
        except (ValueError, TypeError):
            continue

        # Update variables.
        last_fix_word_idx = cur_fix_word_idx
        cur_fix_word_idx = next_fix_word_idx
        cur_fix_dur = next_fix_dur
        if cur_fix_dur is None:
            continue

        next_fix_word_idx = aoi
        next_fix_dur = fixation['duration']

        if next_fix_dur == 0 and not next_fix_word_idx == -1:
            next_fix_word_idx = cur_fix_word_idx

        right_most_word = max(right_most_word, cur_fix_word_idx)

        if cur_fix_word_idx == -1:
            continue

        # Update reading measures for the current word.
        rm_dict[cur_fix_word_idx]['TFT'] += int(cur_fix_dur)
        rm_dict[cur_fix_word_idx]['TFC'] += 1
        if rm_dict[cur_fix_word_idx]['FD'] == 0:
            rm_dict[cur_fix_word_idx]['FD'] += int(cur_fix_dur)

        if right_most_word == cur_fix_word_idx:
            if rm_dict[cur_fix_word_idx]['TRC_out'] == 0:
                rm_dict[cur_fix_word_idx]['FPRT'] += int(cur_fix_dur)
                rm_dict[cur_fix_word_idx]['FPFC'] += 1
                if last_fix_word_idx < cur_fix_word_idx:
                    rm_dict[cur_fix_word_idx]['FFD'] += int(cur_fix_dur)
        else:
            rm_dict[right_most_word]['RPD_exc'] += int(cur_fix_dur)

        if cur_fix_word_idx < last_fix_word_idx:
            rm_dict[cur_fix_word_idx]['TRC_in'] += 1
        if cur_fix_word_idx > next_fix_word_idx:
            rm_dict[cur_fix_word_idx]['TRC_out'] += 1
        if cur_fix_word_idx == right_most_word:
            rm_dict[cur_fix_word_idx]['RBRT'] += int(cur_fix_dur)
        if (
            rm_dict[cur_fix_word_idx]['FRT'] == 0 and
            (not next_fix_word_idx == cur_fix_word_idx or next_fix_dur == 0)
        ):
            rm_dict[cur_fix_word_idx]['FRT'] = rm_dict[cur_fix_word_idx]['TFT']
        if rm_dict[cur_fix_word_idx]['SL_in'] == 0:
            rm_dict[cur_fix_word_idx]['SL_in'] = cur_fix_word_idx - last_fix_word_idx
        if rm_dict[cur_fix_word_idx]['SL_out'] == 0:
            rm_dict[cur_fix_word_idx]['SL_out'] = next_fix_word_idx - cur_fix_word_idx

    # Finalize reading measures.
    rm_list = []
    for aoi_key, aoi_rm in sorted(rm_dict.items()):
        if aoi_key == -1:
            continue
        if aoi_rm['FFD'] == aoi_rm['FPRT']:
            aoi_rm['SFD'] = aoi_rm['FFD']
        aoi_rm['RRT'] = aoi_rm['TFT'] - aoi_rm['FPRT']
        aoi_rm['FPF'] = int(aoi_rm['FFD'] > 0)
        aoi_rm['RR'] = int(aoi_rm['RRT'] > 0)
        aoi_rm['FPReg'] = int(aoi_rm['RPD_exc'] > 0)
        aoi_rm['Fix'] = int(aoi_rm['TFT'] > 0)
        aoi_rm['RPD_inc'] = aoi_rm['RPD_exc'] + aoi_rm['RBRT']

        rm_list.append(aoi_rm)

    return pl.DataFrame(rm_list)

In [10]:
rm_df = compute_reading_measures(
    fixations=events.frame,
    aois=aoi_df,
    word_index_column='aoi_index',
    word_column='content'
)
rm_df.head()

word,word_index,FFD,SFD,FD,FPRT,FPFC,FRT,TFT,RRT,RPD_inc,RPD_exc,RBRT,Fix,FPF,RR,FPReg,TRC_out,TRC_in,SL_in,SL_out,TFC
str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""Last""",0,195,0,195,360,2,360,360,0,360,0,360,1,1,0,0,0,0,1,1,2
"""Wish""",1,109,109,109,109,1,109,109,0,109,0,109,1,1,0,0,0,0,1,1,1
"""Emma""",2,134,134,134,134,1,134,134,0,134,0,134,1,1,0,0,0,0,1,1,1
"""sat""",3,187,187,187,187,1,187,187,0,187,0,187,1,1,0,0,0,0,1,1,1
"""beside""",4,341,341,341,341,1,341,341,0,341,0,341,1,1,0,0,0,0,1,2,1


The resulting DataFrame contains one row per word (Area of Interest, AOI) and subject, along with a range of eye-tracking reading measures.

#### Description of computed reading measures

| Column | Description |
|--------|-------------|
| `word` | The word (AOI) within the text. |
| `word_index` | Zero-based index of the word within the text. |
| `FFD` | **First Fixation Duration** — duration of the first fixation during the first pass only. |
| `SFD` | **Single Fixation Duration** — fixation duration when a word receives exactly one fixation; `0` if the word receives multiple fixations. |
| `FD` | **Fixation Duration** — duration of the first fixation on the word. |
| `FPRT` | **First Pass Reading Time** — sum of all fixation durations on the word during the first pass. |
| `FPFC` | **First Pass Fixation Count** — number of fixations on the word during the first pass. |
| `FRT` | **First Reading Time** — total dwell time from first entering the word until first leaving it. |
| `TFT` | **Total Fixation Time** — total fixation time on the word (`FPRT + RRT`). |
| `RRT` | **Rereading Time** — sum of fixation durations occurring after the first pass. |
| `RPD_inc` | **Regression-Path Duration (inclusive)** — sum of all fixation durations from first entering the word until the first fixation to the right of the word, including fixations on the word itself. |
| `RPD_exc` | **Regression-Path Duration (exclusive)** — time spent on regressed words only, excluding fixations on the current word. |
| `RBRT` | **Right-Bounded Reading Time** — sum of fixation durations on the word before any word to its right is fixated. |
| `Fix` | **Fixation Indicator** — `1` if the word was fixated (`TFT > 0`). |
| `FPF` | **First Pass Fixation Indicator** — `1` if the word was fixated during the first pass (`FFD > 0`). |
| `RR` | **Rereading Indicator** — `1` if the word was reread (`RRT > 0`). |
| `FPReg` | **Regression Indicator** — `1` if regressions occurred (`RPD_exc > 0`). |
| `TRC_out` | **Total Regression Count (Outgoing)** — number of regressions originating from the word. |
| `TRC_in` | **Total Regression Count (Ingoing)** — number of regressions landing on the word. |
| `SL_in` | **Saccade Length (Ingoing)** — word distance between the current word and the previously fixated word at the time of the first fixation on the current word. |
| `SL_out` | **Saccade Length (Outgoing)** — word distance from the current word to the next fixated word, measured at the last fixation of the first reading pass. |
| `TFC` | **Total Fixation Count** — total number of fixations on the word. |
| `subject_id` | Identifier of the subject. |
| `text_id` | Identifier of the text. |

### What you have learned in this tutorial:


- load fixation events into a DataFrame,
- load word-level AOI definitions and their bounding boxes,
- map fixation events to the corresponding AOIs,
- compute word-level reading measures using `compute_reading_measures`, and
- inspect the resulting DataFrame containing the computed reading measures.